In [4]:
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.models import Model

print("All modules imported successfully!")

All modules imported successfully!


In [5]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, emb_size, enc_units):
        super(Encoder, self).__init__()
        self.enc_units = enc_units
        self.embedding = Embedding(vocab_size, emb_size)
        self.lstm = LSTM(enc_units, return_sequences=True, return_state=True)

    def call(self, x):
        x = self.embedding(x)
        output,state_h, state_c = self.lstm(x)
        return output, state_h, state_c
    

In [7]:
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V = Dense(1)
    
    def call(self, query, values):
        query_with_time_axis = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

In [8]:
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units):
        super(Decoder, self).__init__()
        self.dec_units = dec_units
        self.embedding = Embedding(vocab_size, embedding_dim)
        
        self.lstm = LSTM(self.dec_units,
                         return_sequences=True,
                         return_state=True)
        self.fc = Dense(vocab_size)
        self.attention = BahdanauAttention(self.dec_units)

    def call(self, x, hidden, enc_output):
        context_vector, attention_weights = self.attention(hidden, enc_output)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        output, state_h, state_c = self.lstm(x)
        output = tf.reshape(output, (-1, output.shape[2]))
        x = self.fc(output)
        return x, state_h, state_c, attention_weights

In [9]:
BATCH_SIZE = 64
VOCAB_SIZE = 10000     # Size of your dictionary
EMBEDDING_DIM = 256    # Size of the word vectors
UNITS = 1024           # Number of neurons in the LSTMs
SEQUENCE_LENGTH = 20   # e.g., max 20 words per sentence

print("Initializing models...")
encoder = Encoder(VOCAB_SIZE, EMBEDDING_DIM, UNITS)
decoder = Decoder(VOCAB_SIZE, EMBEDDING_DIM, UNITS)

# 3. Create Dummy Data
# Simulating a batch of 64 sentences, each 20 words long
dummy_input = tf.random.uniform((BATCH_SIZE, SEQUENCE_LENGTH), minval=0, maxval=VOCAB_SIZE, dtype=tf.int32)

print("\n--- Testing Encoder ---")
# Pass dummy data through the encoder
sample_encoder_output, sample_hidden, sample_cell = encoder(dummy_input)
print(f"Encoder Output Shape: {sample_encoder_output.shape} (Expected: {BATCH_SIZE}, {SEQUENCE_LENGTH}, {UNITS})")
print(f"Encoder Hidden State Shape: {sample_hidden.shape} (Expected: {BATCH_SIZE}, {UNITS})")

print("\n--- Testing Decoder (Single Timestep) ---")
# Simulating the FIRST decoding step. 
# We pass in a batch of "Start of Sentence" tokens (let's say index 1)
dummy_target_word = tf.random.uniform((BATCH_SIZE, 1), minval=1, maxval=2, dtype=tf.int32)

# Pass the dummy word, the encoder's hidden state, and the encoder's full output into the decoder
sample_decoder_output, dec_hidden, dec_cell, sample_attention_weights = decoder(
    dummy_target_word, 
    sample_hidden, 
    sample_encoder_output
)

print(f"Decoder Output (Logits) Shape: {sample_decoder_output.shape} (Expected: {BATCH_SIZE}, {VOCAB_SIZE})")
print(f"Decoder Hidden State Shape: {dec_hidden.shape} (Expected: {BATCH_SIZE}, {UNITS})")
print(f"Attention Weights Shape: {sample_attention_weights.shape} (Expected: {BATCH_SIZE}, {SEQUENCE_LENGTH}, 1)")

print("\nSuccess! The tensor dimensions align perfectly.")

Initializing models...

--- Testing Encoder ---
Encoder Output Shape: (64, 20, 1024) (Expected: 64, 20, 1024)
Encoder Hidden State Shape: (64, 1024) (Expected: 64, 1024)

--- Testing Decoder (Single Timestep) ---
Decoder Output (Logits) Shape: (64, 10000) (Expected: 64, 10000)
Decoder Hidden State Shape: (64, 1024) (Expected: 64, 1024)
Attention Weights Shape: (64, 20, 1) (Expected: 64, 20, 1)

Success! The tensor dimensions align perfectly.
